# 02 — Model Evaluation

Systematic post-training analysis of the YOLO-OBB boat detection model.

All metric computation is delegated to `src.vessels_detect.vessels_detect.vessels_detect.evaluation.metrics`.
This notebook contains only configuration constants and visualisation code.

| § | Description |
|---|-------------|
| 1 | Setup & constants |
| 2 | Model loading & formal val() metrics |
| 3 | Per-class performance table |
| 4 | Normalised confusion matrix |
| 5 | Dataset statistics |
| 6 | OBB geometric statistics |
| 7 | PR curve |
| 8 | TP / FP / FN detection crops |

In [1]:
%env OPENCV_LOG_LEVEL=ERROR
# ── Standard library ──────────────────────────────────────────────────────
import sys, logging
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

# ── Third-party ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import cv2

# ── Project ───────────────────────────────────────────────────────────────
from src.vessels_detect.utils.config import load_config
from src.vessels_detect.evaluation.metrics import (
    compute_per_class_metrics,
    compute_confusion_matrix,
    build_size_dataframe,
    build_distribution_dataframe,
    parse_label_file,
    match_detections,
    compute_crop_metrics,
    build_crop_summary,
    # § 6 — Systematic Counting Bias
    calculate_counting_bias,
    build_bias_dataframe,
)

logging.getLogger("ultralytics").setLevel(logging.ERROR)

plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "legend.framealpha": 0.7,
    "legend.fontsize": 9,
})
print("Imports OK.")

env: OPENCV_LOG_LEVEL=ERROR
Imports OK.


## 1. Constants

Change these to adapt to a different run or dataset.

In [2]:
MODEL_PATH    = Path("../boat_obb_v33/weights/best.pt")
DATA_PATH     = Path("../data/dataset.yaml")
PROCESSED_DIR = Path("../data/processed")
LABELS_ROOT   = PROCESSED_DIR / "labels"
IMAGES_DIR    = PROCESSED_DIR / "images"
LABELS_DIR    = PROCESSED_DIR / "labels"

SPLITS        = ["train", "val", "test"]

FINAL_CONF    = 0.10
FINAL_IOU     = 0.30
IMG_SIZE      = 640      # must match training imgsz
TILE_SIZE     = 640      # tile size used in preprocessing

print(f"Model  : {MODEL_PATH}")
print(f"Data   : {DATA_PATH}")
print(f"Conf   : {FINAL_CONF}  |  IoU : {FINAL_IOU}")

Model  : ../boat_obb_v33/weights/best.pt
Data   : ../data/dataset.yaml
Conf   : 0.1  |  IoU : 0.3


## 2. Model Loading

In [3]:
from ultralytics import YOLO

model       = YOLO(MODEL_PATH)
CLASS_NAMES = model.names
print("Classes:", CLASS_NAMES)

Classes: {0: 'Pirogue', 1: 'Double_hulled_Pirogue', 2: 'Small_Motorboat', 3: 'Medium_Motorboat', 4: 'Large_Motorboat', 5: 'Sailing_Boat'}


## 3. Formal Evaluation — model.val()

In [17]:
metrics = model.val(
    data=str(DATA_PATH),
    split="test",
    imgsz=IMG_SIZE,
    conf=FINAL_CONF,
    iou=FINAL_IOU,
    save_json=False,
    plots=True,
    verbose=False,
)

print(f"mAP50    : {metrics.box.map50:.4f}")
print(f"mAP50-95 : {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall   : {metrics.box.mr:.4f}")

print(f"Graphs saved to: {metrics.save_dir}")


Ultralytics 8.4.1 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4080, 15937MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 6664.8±1673.0 MB/s, size: 1138.9 KB)
val: Scanning /home/thomas/Documents/test/pleiades-boat-detection/data/processed/labels/test.cache... 6411 images, 6253 backgrounds, 38 corrupt: 100% ━━━━━━━━━━━━ 6411/6411 3.8Git/s 0.0s
val: /home/thomas/Documents/test/pleiades-boat-detection/data/processed/images/test/IMG_PNEO3_STD_202405300712136_MS-FS_ORT_PWOI_000373512_2_2_F_1_RGB_R1C1_15104_2560.tif: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.092055]
val: /home/thomas/Documents/test/pleiades-boat-detection/data/processed/images/test/IMG_PNEO3_STD_202405300712136_MS-FS_ORT_PWOI_000373512_2_2_F_1_RGB_R1C1_15104_2816.tif: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.0230551]
val: /home/thomas/Documents/test/pleiades-boat-detection/data/processed/images/test/IMG_PNEO3_STD_202405300712136_

## 4. Per-Class Performance Table

In [19]:
df_perf = compute_per_class_metrics(metrics, CLASS_NAMES, LABELS_ROOT, test_split="test")
display(df_perf)

,Class,Support,Precision,Recall,F1-score,mAP50,mAP50-95
0,Small_Motorboat,47,0.6539,0.6315,0.6425,0.6165,0.4974
1,Double_hulled_Pirogue,284,0.7024,0.5400,0.6105,0.6171,0.4677
2,Sailing_Boat,17,1.0000,0.1927,0.3231,0.6000,0.3247
3,Pirogue,158,0.1349,0.4685,0.2095,0.2053,0.1263
4,Medium_Motorboat,0,NaN,NaN,NaN,NaN,NaN
5,Large_Motorboat,0,NaN,NaN,NaN,NaN,NaN
6,**Global**,506,0.6228,0.4581,0.5279,0.5097,0.3247


## 5. Normalised Confusion Matrix

In [6]:
metrics_cm = model.val(
    data=str(DATA_PATH), split="test",
    imgsz=IMG_SIZE, conf=FINAL_CONF, iou=FINAL_IOU,
    plots=True, save_json=False, verbose=False,
)

cm_norm, labels = compute_confusion_matrix(metrics_cm, CLASS_NAMES)

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cm_norm, vmin=0, vmax=1, cmap="Blues")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ticks = np.arange(len(labels))
ax.set_xticks(ticks); ax.set_xticklabels(labels, rotation=40, ha="right")
ax.set_yticks(ticks); ax.set_yticklabels(labels)

thresh = cm_norm.max() / 2.0
for i in range(cm_norm.shape[0]):
    for j in range(cm_norm.shape[1]):
        v = cm_norm[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    fontsize=8, color="white" if v > thresh else "black")

ax.set_ylabel("True label"); ax.set_xlabel("Predicted label")
ax.set_title(f"Normalised Confusion Matrix (conf={FINAL_CONF}, IoU={FINAL_IOU})",
             fontweight="bold")
plt.tight_layout()
plt.show()

Ultralytics 8.4.1 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4080, 15937MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 8853.3±1350.7 MB/s, size: 999.4 KB)
val: Scanning /home/thomas/Documents/test/pleiades-boat-detection/data/processed/labels/test.cache... 6411 images, 6253 backgrounds, 38 corrupt: 100% ━━━━━━━━━━━━ 6411/6411 4.5Git/s 0.0s
val: /home/thomas/Documents/test/pleiades-boat-detection/data/processed/images/test/IMG_PNEO3_STD_202405300712136_MS-FS_ORT_PWOI_000373512_2_2_F_1_RGB_R1C1_15104_2560.tif: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.092055]
val: /home/thomas/Documents/test/pleiades-boat-detection/data/processed/images/test/IMG_PNEO3_STD_202405300712136_MS-FS_ORT_PWOI_000373512_2_2_F_1_RGB_R1C1_15104_2816.tif: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.0230551]
val: /home/thomas/Documents/test/pleiades-boat-detection/data/processed/images/test/IMG_PNEO3_STD_202405300712136_M

<Figure size 1350x1200 with 2 Axes>

## 6. Dataset Statistics

In [7]:
df_dist = build_distribution_dataframe(LABELS_ROOT, SPLITS, CLASS_NAMES)
print("Object counts per class and split:")
display(df_dist)

classes = list(CLASS_NAMES.values())
x, width = np.arange(len(classes)), 0.25

fig, ax = plt.subplots(figsize=(12, 5))
for i, split in enumerate(SPLITS):
    col = split.capitalize()
    vals = df_dist[col].values if col in df_dist.columns else [0]*len(classes)
    ax.bar(x + i * width, vals, width, label=col)

ax.set_xticks(x + width)
ax.set_xticklabels(classes, rotation=30, ha="right")
ax.set_ylabel("Object count")
ax.set_title("Class Distribution per Split", fontweight="bold")
ax.legend(); plt.tight_layout(); plt.show()

Object counts per class and split:


,Class,Train,Val,Test,Total
0,Pirogue,1001,232,158,1391
1,Double_hulled_Pirogue,884,123,284,1291
2,Small_Motorboat,511,65,47,623
3,Medium_Motorboat,254,51,0,305
4,Large_Motorboat,81,0,0,81
5,Sailing_Boat,182,16,17,215


<Figure size 1800x750 with 1 Axes>

## 7. OBB Geometric Statistics

Long side, short side, area, and aspect ratio per class across all splits.

In [8]:
df_sizes = build_size_dataframe(LABELS_ROOT, SPLITS, CLASS_NAMES, tile_size=TILE_SIZE)
display(df_sizes)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col, title in zip(
    axes,
    ["Long mean (px)", "Short mean (px)", "Aspect ratio"],
    ["Mean Long Side (px)", "Mean Short Side (px)", "Mean Aspect Ratio"],
):
    ax.barh(df_sizes["Class"], df_sizes[col], color="steelblue")
    ax.set_xlabel(col); ax.set_title(title, fontweight="bold")

plt.tight_layout(); plt.show()

,Class ID,Class,Count,Long mean (px),Long std,Short mean (px),Short std,Area mean (px²),Aspect ratio
0,0,Pirogue,1391,46.67,16.54,13.68,5.72,705.49,3.59
1,1,Double_hulled_Pirogue,1291,75.90,26.79,41.38,15.21,3479.14,1.89
2,2,Small_Motorboat,623,75.16,24.82,27.20,8.54,2194.96,2.83
3,3,Medium_Motorboat,305,253.81,149.95,60.19,34.36,20051.48,4.27
4,4,Large_Motorboat,81,789.74,476.44,201.56,132.13,219685.00,4.09
5,5,Sailing_Boat,215,128.71,38.34,38.14,13.96,4967.25,3.62


<Figure size 2250x750 with 3 Axes>

## 8. PR Curve

Sweep confidence threshold and record precision/recall to plot the PR curve.

In [9]:
thresholds  = np.linspace(0.05, 0.50, 10)
pr_records  = []

images_test = sorted((IMAGES_DIR / "test").glob("*.tif"))
print(f"Running PR sweep over {len(thresholds)} thresholds on {len(images_test)} tiles …")

for conf in thresholds:
    tp = fp = fn = 0
    for img_path in images_test:
        import rasterio as _rio
        with _rio.open(img_path) as ds:
            w, h = ds.width, ds.height

        lbl   = LABELS_ROOT / "test" / (img_path.stem + ".txt")
        gt    = parse_label_file(lbl, w, h)
        res   = model.predict(str(img_path), imgsz=IMG_SIZE,
                              conf=float(conf), iou=FINAL_IOU, verbose=False)[0]

        preds = []
        if res.obb is not None and len(res.obb):
            for pts, cid, cf in zip(
                res.obb.xyxyxyxy.cpu().numpy(),
                res.obb.cls.cpu().numpy().astype(int),
                res.obb.conf.cpu().numpy(),
            ):
                preds.append((int(cid), pts.astype(np.float32), float(cf)))

        m_pred, m_gt = match_detections(gt, preds, iou_threshold=FINAL_IOU)
        tp += sum(m_pred)
        fp += sum(not x for x in m_pred)
        fn += sum(not x for x in m_gt)

    p, r, f1 = compute_crop_metrics(tp, fp, fn)
    pr_records.append({"conf": conf, "precision": p, "recall": r, "f1": f1})

df_pr = pd.DataFrame(pr_records)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(df_pr["recall"], df_pr["precision"], "b-o", ms=4, label="P-R curve")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title("Precision-Recall Curve", fontweight="bold")
best = df_pr.loc[df_pr["f1"].idxmax()]
ax.annotate(f'F1={best.f1:.3f}\nconf={best.conf:.2f}',
            xy=(best.recall, best.precision), fontsize=9,
            xytext=(best.recall - 0.2, best.precision - 0.12),
            arrowprops=dict(arrowstyle="->")); ax.legend()

ax = axes[1]
ax.plot(df_pr["conf"], df_pr["precision"], label="Precision")
ax.plot(df_pr["conf"], df_pr["recall"],    label="Recall")
ax.plot(df_pr["conf"], df_pr["f1"],        label="F1", linestyle="--")
ax.set_xlabel("Confidence threshold"); ax.set_ylabel("Score")
ax.set_title("Metrics vs Confidence Threshold", fontweight="bold")
ax.legend()

plt.tight_layout(); plt.show()

Running PR sweep over 10 thresholds on 6411 tiles …


KeyboardInterrupt: 

## 9. Detection Crop Export — TP / FP / FN

Crops are saved to `crops/<TruePositive|FalsePositive|FalseNegative>/<class>/`.
A summary CSV is written to `crops/summary.csv`.

In [ ]:
IOU_MATCH  = FINAL_IOU
CROP_PAD   = 80
CROPS_ROOT = Path("../crops")
SPLIT      = "test"

OBB_COLOR     = (0, 0, 255)
OBB_THICKNESS = 2

for sub in ("TruePositive", "FalsePositive", "FalseNegative"):
    (CROPS_ROOT / sub).mkdir(parents=True, exist_ok=True)


def crop_obb(img_bgr, pts_px, pad=CROP_PAD):
    h, w = img_bgr.shape[:2]
    x1 = max(0,     int(pts_px[:, 0].min()) - pad)
    y1 = max(0,     int(pts_px[:, 1].min()) - pad)
    x2 = min(w - 1, int(pts_px[:, 0].max()) + pad)
    y2 = min(h - 1, int(pts_px[:, 1].max()) + pad)
    return img_bgr[y1:y2, x1:x2].copy(), (x1, y1)

def draw_obb(img, pts_px, origin=(0, 0)):
    shifted = pts_px.copy()
    shifted[:, 0] -= origin[0]; shifted[:, 1] -= origin[1]
    cv2.polylines(img, [shifted.astype(np.int32).reshape(-1, 1, 2)],
                  True, OBB_COLOR, OBB_THICKNESS)
    return img


summary_rows = []
tp_count = fp_count = fn_count = 0

for img_path in sorted((IMAGES_DIR / SPLIT).glob("*.tif")):
    import rasterio as _rio
    with _rio.open(img_path) as ds:
        rgb_data = ds.read()
    img_bgr = cv2.cvtColor(rgb_data[:3].transpose(1, 2, 0), cv2.COLOR_RGB2BGR)
    h, w = img_bgr.shape[:2]
    stem  = img_path.stem

    lbl = LABELS_ROOT / SPLIT / (stem + ".txt")
    gt  = parse_label_file(lbl, w, h)

    res   = model.predict(str(img_path), imgsz=IMG_SIZE,
                          conf=FINAL_CONF, iou=FINAL_IOU, verbose=False)[0]
    preds = []
    if res.obb is not None and len(res.obb):
        for pts, cid, cf in zip(
            res.obb.xyxyxyxy.cpu().numpy(),
            res.obb.cls.cpu().numpy().astype(int),
            res.obb.conf.cpu().numpy(),
        ):
            preds.append((int(cid), pts.astype(np.float32), float(cf)))

    m_pred, m_gt = match_detections(gt, preds, IOU_MATCH)

    # TPs
    for pi, (pr_cid, pr_pts, pr_cf) in enumerate(preds):
        if m_pred[pi]:
            crop, origin = crop_obb(img_bgr, pr_pts)
            if crop.size:
                crop = draw_obb(crop, pr_pts, origin)
                out  = CROPS_ROOT / "TruePositive" / CLASS_NAMES[pr_cid]
                out.mkdir(parents=True, exist_ok=True)
                cv2.imwrite(str(out / f"{stem}__tp__{tp_count:04d}.png"), crop)
                summary_rows.append({"image": stem, "category": "TruePositive",
                                     "class": CLASS_NAMES[pr_cid], "index": tp_count})
                tp_count += 1

    # FPs
    for pi, (pr_cid, pr_pts, pr_cf) in enumerate(preds):
        if not m_pred[pi]:
            out = CROPS_ROOT / "FalsePositive" / CLASS_NAMES[pr_cid]
            out.mkdir(parents=True, exist_ok=True)
            canvas = img_bgr.copy()
            draw_obb(canvas, pr_pts)
            cv2.imwrite(str(out / f"{stem}__fp__{fp_count:04d}.png"), canvas)
            summary_rows.append({"image": stem, "category": "FalsePositive",
                                 "class": CLASS_NAMES[pr_cid], "index": fp_count})
            fp_count += 1

    # FNs
    for gi, (gt_cid, gt_pts) in enumerate(gt):
        if not m_gt[gi]:
            crop, _ = crop_obb(img_bgr, gt_pts, pad=CROP_PAD)
            if crop.size:
                out = CROPS_ROOT / "FalseNegative" / CLASS_NAMES[gt_cid]
                out.mkdir(parents=True, exist_ok=True)
                cv2.imwrite(str(out / f"{stem}__fn__{fn_count:04d}.png"), crop)
                summary_rows.append({"image": stem, "category": "FalseNegative",
                                     "class": CLASS_NAMES[gt_cid], "index": fn_count})
                fn_count += 1

df_crops = build_crop_summary(summary_rows)
if not df_crops.empty:
    df_crops.to_csv(CROPS_ROOT / "summary.csv", index=False)

p, r, f1 = compute_crop_metrics(tp_count, fp_count, fn_count)
print(f"TP: {tp_count}  FP: {fp_count}  FN: {fn_count}")
print(f"Precision: {p:.4f}  Recall: {r:.4f}  F1: {f1:.4f}")
print(f"Crops saved to: {CROPS_ROOT.resolve()}")
display(df_crops)

TP: 323  FP: 1083  FN: 183
Precision: 0.2297  Recall: 0.6383  F1: 0.3379
Crops saved to: /home/thomas/Documents/test/pleiades-boat-detection/crops


,category,class,count
0,FalseNegative,Double_hulled_Pirogue,95
1,FalseNegative,Pirogue,66
2,FalseNegative,Sailing_Boat,7
3,FalseNegative,Small_Motorboat,15
4,FalsePositive,Double_hulled_Pirogue,132
5,FalsePositive,Medium_Motorboat,1
6,FalsePositive,Pirogue,922
7,FalsePositive,Sailing_Boat,2
8,FalsePositive,Small_Motorboat,26
9,TruePositive,Double_hulled_Pirogue,166


## 10. Systematic Counting Bias

Measures whether the model **systematically** over- or under-counts each class across the entire evaluation split, expressed as a percentage relative to the ground-truth total:

```
Bias (%) = ((ΣPredicted − ΣGroundTruth) / ΣGroundTruth) × 100
```

| Value | Interpretation |
|-------|---------------|
| Negative | Model **undercounts** — confidence threshold may be too high |
| Positive | Model **overcounts** — likely duplicate detections or low threshold |
| ≈ 0 %  | Aggregate counts are well-calibrated |

> **Note:** This is a dataset-level aggregate metric.  Individual image counts can still be noisy even when the overall bias is near zero.

In [ ]:
# ── 10.1  Extract per-image, per-class GT and predicted counts ────────────────
#
# Strategy: iterate over every tile in the test split.
#   GT counts  → read directly from the YOLO OBB label files.
#   Pred counts → run model.predict() tile-by-tile (reuses the already-loaded
#                 model; no second model.val() call needed).
#
# Result structure:
#   gt_counts_per_class   = { class_id: [count_img0, count_img1, …] }
#   pred_counts_per_class = { class_id: [count_img0, count_img1, …] }
# Both dicts are aligned: index i refers to the same image in every class list.

EVAL_SPLIT  = "test"
images_test = sorted((IMAGES_DIR / EVAL_SPLIT).glob("*.tif"))
labels_test = LABELS_DIR / EVAL_SPLIT       # YOLO label .txt files

n_classes   = len(CLASS_NAMES)
class_ids   = list(CLASS_NAMES.keys())      # e.g. [0, 1, 2, 3, 4, 5]

gt_counts_per_class:   dict[int, list[int]] = {cid: [] for cid in class_ids}
pred_counts_per_class: dict[int, list[int]] = {cid: [] for cid in class_ids}

for img_path in images_test:
    stem = img_path.stem

    # ── Ground truth ──────────────────────────────────────────────────────
    from collections import Counter as _Counter
    gt_for_img: _Counter[int] = _Counter()
    lbl_path = labels_test / f"{stem}.txt"
    if lbl_path.exists():
        for line in lbl_path.read_text().splitlines():
            line = line.strip()
            if line:
                gt_for_img[int(line.split()[0])] += 1

    # ── Predictions ───────────────────────────────────────────────────────
    results      = model.predict(
        source=str(img_path),
        imgsz=IMG_SIZE,
        conf=FINAL_CONF,
        iou=FINAL_IOU,
        verbose=False,
    )
    pred_for_img: _Counter[int] = _Counter()
    if results and results[0].obb is not None:
        for cid in results[0].obb.cls.int().tolist():
            pred_for_img[cid] += 1

    # ── Accumulate ────────────────────────────────────────────────────────
    for cid in class_ids:
        gt_counts_per_class[cid].append(gt_for_img.get(cid, 0))
        pred_counts_per_class[cid].append(pred_for_img.get(cid, 0))

print(f"Processed {len(images_test)} test tiles.")

In [ ]:
# ── 10.2  Counting Bias summary table ────────────────────────────────────────

df_bias = build_bias_dataframe(
    gt_counts_per_class=gt_counts_per_class,
    pred_counts_per_class=pred_counts_per_class,
    class_names=CLASS_NAMES,
)

display(df_bias)

In [ ]:
# ── 10.3  Counting Bias bar chart ─────────────────────────────────────────────

def _parse_bias_pct(bias_str: str) -> float | None:
    """Convert a formatted bias string back to float for plotting.

    Returns None for "inf" entries so they can be annotated separately.
    """
    if bias_str == "inf":
        return None
    return float(bias_str.rstrip("%"))


def plot_counting_bias(df: pd.DataFrame) -> plt.Figure:
    """Render a signed bar chart of Systematic Counting Bias per class.

    Args:
        df: DataFrame produced by :func:`build_bias_dataframe`.
            Must contain ``"Class"`` and ``"Bias (%)"`` columns.

    Returns:
        The :class:`~matplotlib.figure.Figure` object.
    """
    # Drop the aggregate row for the per-class chart.
    plot_df = df[df["Class"] != "All Classes"].copy()
    plot_df["_bias_float"] = plot_df["Bias (%)"].apply(_parse_bias_pct)

    classes      = plot_df["Class"].tolist()
    bias_values  = plot_df["_bias_float"].tolist()
    bar_colors   = [
        "#d62728" if (v is not None and v < 0) else "#2196F3"
        for v in bias_values
    ]

    fig, ax = plt.subplots(figsize=(10, 5))

    bars = ax.bar(
        x=range(len(classes)),
        height=[v if v is not None else 0 for v in bias_values],
        color=bar_colors,
        edgecolor="white",
        linewidth=0.6,
        zorder=3,
    )

    # Perfect-counting reference line.
    ax.axhline(y=0, color="black", linewidth=1.2, linestyle="--", zorder=4)

    # Annotate each bar with the formatted bias string.
    for i, (bar, label) in enumerate(zip(bars, plot_df["Bias (%)"].tolist())):
        bias_float = bias_values[i]
        y_anchor   = bar.get_height()
        v_offset   = 0.8 if (bias_float is None or bias_float >= 0) else -0.8
        va         = "bottom" if (bias_float is None or bias_float >= 0) else "top"
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            y_anchor + v_offset,
            label,
            ha="center", va=va,
            fontsize=9, fontweight="bold",
        )

    ax.set_xticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=20, ha="right")
    ax.set_ylabel("Counting Bias (%)")
    ax.set_title(
        f"Systematic Counting Bias per Class — {EVAL_SPLIT.capitalize()} split\n"
        f"conf={FINAL_CONF}  iou={FINAL_IOU}",
        pad=10,
    )

    # Colour legend.
    from matplotlib.patches import Patch
    legend_handles = [
        Patch(facecolor="#d62728", label="Undercounting (Bias < 0)"),
        Patch(facecolor="#2196F3", label="Overcounting  (Bias ≥ 0)"),
    ]
    ax.legend(handles=legend_handles, loc="upper right")

    fig.tight_layout()
    return fig


fig_bias = plot_counting_bias(df_bias)
plt.show()